In [5]:
import pandas as pd
import numpy as np
import gc # Garbage Collection
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import re
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Function to process application_train and application_test
def process_application(num_rows=None):
    df = pd.read_csv('./home-credit-default-risk/application_train.csv', nrows=num_rows)
    test_df = pd.read_csv('./home-credit-default-risk/application_test.csv', nrows=num_rows)
    df = pd.concat([df, test_df], ignore_index=True)

    # Clean up DAYS_EMPLOYED
    df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

    # Domain-specific features
    df['CREDIT_INCOME_PERCENT'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['ANNUITY_INCOME_PERCENT'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['DAYS_EMPLOYED_PERCENT'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    
    # One-hot encode categorical features
    df, cat_cols = one_hot_encoder(df)
    
    return df

# Function to one-hot encode
def one_hot_encoder(df, nan_as_category=True):
    original_columns = list(df.columns)
    categorical_columns = [col for col in df.columns if df[col].dtype == 'object']
    df = pd.get_dummies(df, columns=categorical_columns, dummy_na=nan_as_category)
    new_columns = [c for c in df.columns if c not in original_columns]
    return df, new_columns

# Function to process bureau and bureau_balance
def process_bureau(num_rows=None):
    bureau = pd.read_csv('./home-credit-default-risk/bureau.csv', nrows=num_rows)
    bb = pd.read_csv('./home-credit-default-risk/bureau_balance.csv', nrows=num_rows)
    
    # Process bureau_balance
    bb, bb_cat = one_hot_encoder(bb, nan_as_category=True)
    bb_aggregations = {'MONTHS_BALANCE': ['min', 'max', 'size']}
    for col in bb_cat:
        bb_aggregations[col] = ['mean']
    bb_agg = bb.groupby('SK_ID_BUREAU').agg(bb_aggregations)
    bb_agg.columns = pd.Index([e[0] + "_" + e[1].upper() for e in bb_agg.columns.tolist()])
    
    # Merge with bureau
    bureau = bureau.join(bb_agg, how='left', on='SK_ID_BUREAU')
    bureau.drop(['SK_ID_BUREAU'], axis=1, inplace=True)
    del bb, bb_agg
    gc.collect()
    
    # Bureau aggregations
    num_aggregations = {
        'DAYS_CREDIT': ['min', 'max', 'mean', 'var'],
        'CREDIT_DAY_OVERDUE': ['max', 'mean'],
        'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
    }
    
    bureau_agg = bureau.groupby('SK_ID_CURR').agg({**num_aggregations})
    bureau_agg.columns = pd.Index(['BURO_' + e[0] + "_" + e[1].upper() for e in bureau_agg.columns.tolist()])
    
    # Count of active/closed credits
    active = bureau[bureau['CREDIT_ACTIVE'] == 'Active']
    active_agg = active.groupby('SK_ID_CURR').size().reset_index(name='BURO_ACTIVE_COUNT')
    bureau_agg = bureau_agg.join(active_agg.set_index('SK_ID_CURR'), how='left')
    bureau_agg['BURO_ACTIVE_COUNT'].fillna(0, inplace=True)
    
    return bureau_agg

# Function to process previous_applications
def process_previous_applications(num_rows=None):
    prev = pd.read_csv('./home-credit-default-risk/previous_application.csv', nrows=num_rows)
    prev, cat_cols = one_hot_encoder(prev, nan_as_category=True)
    
    # Days 365.243 values -> nan
    prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace=True)
    prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace=True)
    prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace=True)
    prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace=True)
    prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace=True)
    
    # Add feature: value ask / value received percentage
    prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']
    
    # Aggregations
    aggregations = {
        'AMT_ANNUITY': ['min', 'max', 'mean'],
        'AMT_APPLICATION': ['min', 'max', 'mean'],
        'AMT_CREDIT': ['min', 'max', 'mean'],
        'APP_CREDIT_PERC': ['min', 'max', 'mean', 'var'],
        'AMT_DOWN_PAYMENT': ['min', 'max', 'mean'],
        'RATE_DOWN_PAYMENT': ['min', 'max', 'mean'],
        'DAYS_DECISION': ['min', 'max', 'mean'],
        'CNT_PAYMENT': ['mean', 'sum'],
    }
    prev_agg = prev.groupby('SK_ID_CURR').agg(aggregations)
    prev_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_agg.columns.tolist()])
    
    # Approved applications aggregations
    approved = prev[prev['NAME_CONTRACT_STATUS_Approved'] == 1]
    approved_agg = approved.groupby('SK_ID_CURR').agg({'SK_ID_PREV': ['count']})
    approved_agg.columns = pd.Index(['PREV_APPROVED_COUNT'])
    prev_agg = prev_agg.join(approved_agg, how='left')

    # Refused applications aggregations
    refused = prev[prev['NAME_CONTRACT_STATUS_Refused'] == 1]
    refused_agg = refused.groupby('SK_ID_CURR').agg({'SK_ID_PREV': ['count']})
    refused_agg.columns = pd.Index(['PREV_REFUSED_COUNT'])
    prev_agg = prev_agg.join(refused_agg, how='left')
    
    return prev_agg

# Function to process installments_payments
def process_installments_payments(num_rows=None):
    ins = pd.read_csv('./home-credit-default-risk/installments_payments.csv', nrows=num_rows)
    
    # Create features
    ins['PAYMENT_PERC'] = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT']
    ins['PAYMENT_DIFF'] = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
    ins['DPD'] = ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']
    ins['DBD'] = ins['DAYS_INSTALMENT'] - ins['DAYS_ENTRY_PAYMENT']
    ins['DPD'] = ins['DPD'].apply(lambda x: x if x > 0 else 0)
    ins['DBD'] = ins['DBD'].apply(lambda x: x if x > 0 else 0)
    
    # Aggregations
    aggregations = {
        'NUM_INSTALMENT_VERSION': ['nunique'],
        'DPD': ['max', 'mean', 'sum'],
        'DBD': ['max', 'mean', 'sum'],
        'PAYMENT_PERC': ['max', 'mean', 'sum', 'var'],
        'PAYMENT_DIFF': ['max', 'mean', 'sum', 'var'],
        'AMT_INSTALMENT': ['max', 'mean', 'sum'],
        'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
        'DAYS_ENTRY_PAYMENT': ['max', 'mean', 'sum']
    }
    
    ins_agg = ins.groupby('SK_ID_CURR').agg(aggregations)
    ins_agg.columns = pd.Index(['INSTAL_' + e[0] + "_" + e[1].upper() for e in ins_agg.columns.tolist()])
    
    # Count installments
    ins_agg['INSTAL_COUNT'] = ins.groupby('SK_ID_CURR').size()
    
    return ins_agg

# ----- Main Execution -----
df = process_application()
bureau = process_bureau()
print("Bureau df shape:", bureau.shape)
df = df.join(bureau, how='left', on='SK_ID_CURR')
del bureau; gc.collect()

prev = process_previous_applications()
print("Previous applications df shape:", prev.shape)
df = df.join(prev, how='left', on='SK_ID_CURR')
del prev; gc.collect()

ins = process_installments_payments()
print("Installments payments df shape:", ins.shape)
df = df.join(ins, how='left', on='SK_ID_CURR')
del ins; gc.collect()

# Note: POS_CASH and credit_card_balance can be processed with similar aggregation logic
# and joined as well for a complete feature set.

print("Final DataFrame shape:", df.shape)

Bureau df shape: (305811, 13)
Previous applications df shape: (338857, 26)
Installments payments df shape: (339587, 26)
Final DataFrame shape: (356255, 331)


In [7]:
# Function to train the model and make predictions
def kfold_lightgbm(df, num_folds=5, stratified=True):
    train_df = df[df['TARGET'].notnull()]
    test_df = df[df['TARGET'].isnull()]
    print(f"Starting LightGBM. Train shape: {train_df.shape}, test shape: {test_df.shape}")
    
    del df
    gc.collect()

    # Sanitize column names for LightGBM
    train_df = train_df.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    test_df = test_df.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
    
    # Cross validation model
    if stratified:
        folds = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=47)
    else:
        folds = KFold(n_splits=num_folds, shuffle=True, random_state=47)
        
    # Create arrays and dataframes to store results
    oof_preds = np.zeros(train_df.shape[0])
    sub_preds = np.zeros(test_df.shape[0])
    feature_importance_df = pd.DataFrame()
    feats = [f for f in train_df.columns if f not in ['TARGET','SK_ID_CURR','SK_ID_BUREAU','SK_ID_PREV']]
    
    for n_fold, (train_idx, valid_idx) in enumerate(folds.split(train_df[feats], train_df['TARGET'])):
        train_x, train_y = train_df[feats].iloc[train_idx], train_df['TARGET'].iloc[train_idx]
        valid_x, valid_y = train_df[feats].iloc[valid_idx], train_df['TARGET'].iloc[valid_idx]

        # LightGBM parameters (tuned for this competition)
        clf = lgb.LGBMClassifier(
            n_estimators=10000,
            learning_rate=0.02,
            num_leaves=34,
            colsample_bytree=0.9497036,
            subsample=0.8715623,
            max_depth=8,
            reg_alpha=0.041545473,
            reg_lambda=0.0735294,
            min_split_gain=0.0222415,
            min_child_weight=39.3259775,
            silent=-1,
            verbose=-1, )

        # The 'callbacks' argument replaces verbose and early_stopping_rounds
        clf.fit(train_x, train_y, 
                eval_set=[(valid_x, valid_y)], # It's best practice to only evaluate on the validation set for early stopping
                eval_metric='auc', 
                callbacks=[lgb.early_stopping(stopping_rounds=200),
                        lgb.log_evaluation(period=200)])

        oof_preds[valid_idx] = clf.predict_proba(valid_x, num_iteration=clf.best_iteration_)[:, 1]
        sub_preds += clf.predict_proba(test_df[feats], num_iteration=clf.best_iteration_)[:, 1] / folds.n_splits

        print(f'Fold {n_fold + 1} AUC : {roc_auc_score(valid_y, oof_preds[valid_idx])}')
        del clf, train_x, train_y, valid_x, valid_y
        gc.collect()

    print(f'Full OOF AUC score {roc_auc_score(train_df["TARGET"], oof_preds)}')
    
    # Write submission file
    test_df['TARGET'] = sub_preds
    submission = test_df[['SK_ID_CURR', 'TARGET']]
    submission.to_csv('submission_lgbm.csv', index=False)
    print("Submission file created.")

# ----- Execute Modeling -----
kfold_lightgbm(df)

Starting LightGBM. Train shape: (307511, 331), test shape: (48744, 331)
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.770557	valid_0's binary_logloss: 0.243214
[400]	valid_0's auc: 0.78081	valid_0's binary_logloss: 0.239303
[600]	valid_0's auc: 0.783961	valid_0's binary_logloss: 0.238185
[800]	valid_0's auc: 0.785298	valid_0's binary_logloss: 0.237718
[1000]	valid_0's auc: 0.785971	valid_0's binary_logloss: 0.237471
[1200]	valid_0's auc: 0.786114	valid_0's binary_logloss: 0.237389
[1400]	valid_0's auc: 0.786294	valid_0's binary_logloss: 0.237308
[1600]	valid_0's auc: 0.786318	valid_0's binary_logloss: 0.237294
Early stopping, best iteration is:
[1409]	valid_0's auc: 0.786322	valid_0's binary_logloss: 0.237299
Fold 1 AUC : 0.7863217698105851
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.770484	valid_0's binary_logloss: 0.243027
[400]	valid_0's auc: 0.78023	valid_0's binary_logloss: 0.239199
[600]	valid_0's auc